# The Bias–Variance Tradeoff

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/03-bias-variance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Simulating the decomposition

Bias and variance are defined over *many* training sets. Simulation lets us actually draw them.

In [ ]:
def true_f(x): return np.sin(2 * np.pi * x)

def knn_predict(Xtr, ytr, xq, k):
    idx = np.argsort(np.abs(Xtr[None, :] - xq[:, None]), axis=1)[:, :k]
    return ytr[idx].mean(axis=1)

x_grid = np.linspace(0, 1, 200)
def draw_dataset(n=40, noise=0.3):
    x = np.sort(np.random.rand(n)); return x, true_f(x) + noise * np.random.randn(n)

## Many datasets, two values of k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in [(axes[0], 1), (axes[1], 25)]:
    preds = []
    for _ in range(30):
        x, y = draw_dataset()
        p = knn_predict(x, y, x_grid, k); preds.append(p)
        ax.plot(x_grid, p, color='#6366f1', alpha=0.15, lw=1)
    ax.plot(x_grid, true_f(x_grid), color='#14b8a6', lw=2, label='truth')
    ax.plot(x_grid, np.mean(preds, axis=0), color='#f43f5e', lw=2, ls='--', label='avg prediction')
    ax.set_title(f'k = {k}'); ax.legend()
plt.tight_layout(); plt.show()
# k=1: the cloud of fits is wide (variance) but centered on truth (low bias)
# k=25: every fit is nearly identical (low variance) but flattened (bias)

## The U-shaped error curve

In [ ]:
ks = range(1, 31)
bias2, var = [], []
for k in ks:
    preds = np.array([knn_predict(*draw_dataset(), x_grid, k) for _ in range(100)])
    bias2.append(np.mean((preds.mean(0) - true_f(x_grid)) ** 2))
    var.append(preds.var(0).mean())
bias2, var = np.array(bias2), np.array(var)

plt.figure(figsize=(7, 4.5))
plt.plot(ks, bias2, color='#facc15', lw=2, label='bias²')
plt.plot(ks, var, color='#6366f1', lw=2, label='variance')
plt.plot(ks, bias2 + var, color='#f43f5e', lw=2.5, label='bias² + variance')
plt.axvline(ks[np.argmin(bias2 + var)], color='#14b8a6', ls=':', label='best k')
plt.xlabel('k'); plt.ylabel('error'); plt.legend(); plt.show()

**Try it:** increase the noise to 0.6 — the best k moves up (more averaging needed). Shrink the dataset to n=15 — same direction. The optimal complexity depends on data size and noise, never on the model alone.